# Noise Modeling & Image Restoration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import time

### Load and Prepare Grayscale Image


In [ ]:
IMAGE_PATH = "./images/Task_2_Image_Restoration/grayscale.jpg"   # Change this path if needed
# IMAGE_PATH = "./images/Task_2_Image_Restoration/5226523.jpg"
# IMAGE_PATH = "./images/Task_2_Image_Restoration/kP0u2.jpg"

def load_grayscale_image(path):
    """
    Loads an image as grayscale and returns it as a float64 NumPy array.
    Pixel range: [0, 255]
    """
    img = Image.open(path).convert("L")
    img = np.array(img, dtype=np.float64)
    return img

original = load_grayscale_image(IMAGE_PATH)

plt.figure(figsize=(5, 5))
plt.imshow(original, cmap="gray", vmin=0, vmax=255)
plt.title("Original Clean Grayscale Image")
plt.axis("off")
plt.show()

print("Image shape:", original.shape)
print("Min pixel value:", original.min())
print("Max pixel value:", original.max())


## 2.1 Noise Generation
### Gaussian Noise and Salt & Pepper Noise

In [ ]:
def add_gaussian_noise(image, variance, seed=None):
    """
    Adds Gaussian noise with mean 0 and given variance.
    
    η ~ N(0, σ²)
    where variance = σ²
    """
    if seed is not None:
        np.random.seed(seed)
        
    sigma = np.sqrt(variance)
    noise = np.random.normal(loc=0, scale=sigma, size=image.shape)
    noisy = image + noise
    
    # Clip to valid grayscale range
    noisy = np.clip(noisy, 0, 255)
    return noisy


def add_salt_pepper_noise(image, density, seed=None):
    """
    Adds Salt & Pepper noise.
    
    density controls the percentage of corrupted pixels.
    Half of corrupted pixels become 0, half become 255.
    """
    if seed is not None:
        np.random.seed(seed)
        
    noisy = image.copy()
    random_matrix = np.random.rand(*image.shape)
    
    pepper_mask = random_matrix < (density / 2)
    salt_mask = random_matrix > (1 - density / 2)
    
    noisy[pepper_mask] = 0
    noisy[salt_mask] = 255
    
    return noisy


### Generate Noisy Images

In [ ]:
gaussian_variances = [100, 400, 900]
sp_densities = [0.02, 0.05, 0.10]

noisy_images = {}

for var in gaussian_variances:
    key = f"Gaussian_var_{var}"
    noisy_images[key] = add_gaussian_noise(original, variance=var, seed=42)

for density in sp_densities:
    key = f"SaltPepper_density_{density}"
    noisy_images[key] = add_salt_pepper_noise(original, density=density, seed=42)

plt.figure(figsize=(15, 8))

plt.subplot(2, 4, 1)
plt.imshow(original, cmap="gray", vmin=0, vmax=255)
plt.title("Original")
plt.axis("off")

for idx, (name, img) in enumerate(noisy_images.items(), start=2):
    plt.subplot(2, 4, idx)
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.show()


### Padding Functions

In [ ]:
def zero_padding(image, pad_h, pad_w):
    """
    Pads the image with zeros.
    """
    H, W = image.shape
    padded = np.zeros((H + 2 * pad_h, W + 2 * pad_w), dtype=np.float64)
    padded[pad_h:pad_h + H, pad_w:pad_w + W] = image
    return padded


def mirror_padding(image, pad_h, pad_w):
    """
    Pads the image using edge replication.
    The outermost pixels are duplicated.
    
    This is sometimes also called edge padding.
    """
    H, W = image.shape
    padded = np.zeros((H + 2 * pad_h, W + 2 * pad_w), dtype=np.float64)
    
    # Center
    padded[pad_h:pad_h + H, pad_w:pad_w + W] = image
    
    # Top and bottom replicated rows
    for i in range(pad_h):
        padded[i, pad_w:pad_w + W] = image[0, :]
        padded[pad_h + H + i, pad_w:pad_w + W] = image[H - 1, :]
    
    # Left and right replicated columns
    for j in range(pad_w):
        padded[:, j] = padded[:, pad_w]
        padded[:, pad_w + W + j] = padded[:, pad_w + W - 1]
    
    return padded


def pad_image(image, pad_h, pad_w, mode="mirror"):
    """
    Wrapper function for choosing padding method.
    """
    if mode == "zero":
        return zero_padding(image, pad_h, pad_w)
    elif mode == "mirror":
        return mirror_padding(image, pad_h, pad_w)
    else:
        raise ValueError("Padding mode must be either 'zero' or 'mirror'.")


## 2.2 Image Restoration
### Manual Convolution Function

In [ ]:
def manual_convolution(image, kernel, padding_mode="mirror"):
    """
    Performs 2D convolution manually.
    
    image: 2D grayscale image
    kernel: 2D filter kernel
    padding_mode: 'zero' or 'mirror'
    """
    image = image.astype(np.float64)
    kernel = np.array(kernel, dtype=np.float64)
    
    H, W = image.shape
    kH, kW = kernel.shape
    
    pad_h = kH // 2
    pad_w = kW // 2
    
    padded = pad_image(image, pad_h, pad_w, mode=padding_mode)
    
    output = np.zeros_like(image, dtype=np.float64)
    
    # Flip kernel for true convolution
    flipped_kernel = np.flipud(np.fliplr(kernel))
    
    for i in range(H):
        for j in range(W):
            region = padded[i:i + kH, j:j + kW]
            output[i, j] = np.sum(region * flipped_kernel)
    
    return output


### Mean Filter

In [ ]:
def mean_filter(image, kernel_size=3, padding_mode="mirror"):
    """
    Mean filter from scratch.
    Every pixel is replaced by the average of its neighborhood.
    """
    kernel = np.ones((kernel_size, kernel_size), dtype=np.float64)
    kernel = kernel / (kernel_size * kernel_size)
    
    filtered = manual_convolution(image, kernel, padding_mode=padding_mode)
    return np.clip(filtered, 0, 255)

### Median Filter

In [ ]:
def median_filter(image, kernel_size=3, padding_mode="mirror"):
    """
    Median filter from scratch.
    Every pixel is replaced by the median value of its neighborhood.
    """
    image = image.astype(np.float64)
    H, W = image.shape
    
    pad = kernel_size // 2
    padded = pad_image(image, pad, pad, mode=padding_mode)
    
    output = np.zeros_like(image, dtype=np.float64)
    
    for i in range(H):
        for j in range(W):
            region = padded[i:i + kernel_size, j:j + kernel_size]
            output[i, j] = np.median(region)
    
    return np.clip(output, 0, 255)


### Gaussian Filter

In [ ]:
def create_gaussian_kernel(kernel_size=5, sigma=1.0):
    """
    Creates a Gaussian kernel manually.
    
    G(x, y) = 1 / (2πσ²) * exp(-(x² + y²) / (2σ²))
    """
    center = kernel_size // 2
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float64)
    
    for i in range(kernel_size):
        for j in range(kernel_size):
            x = i - center
            y = j - center
            kernel[i, j] = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    
    # Normalize so the kernel sum equals 1
    kernel = kernel / np.sum(kernel)
    return kernel


def gaussian_filter(image, kernel_size=5, sigma=1.0, padding_mode="mirror"):
    """
    Gaussian smoothing filter from scratch.
    """
    kernel = create_gaussian_kernel(kernel_size=kernel_size, sigma=sigma)
    filtered = manual_convolution(image, kernel, padding_mode=padding_mode)
    return np.clip(filtered, 0, 255)


### Apply All Filters to All Noisy Images

In [ ]:
padding_used = "mirror"

restored_images = {}
execution_times = []

for noise_name, noisy_img in noisy_images.items():
    restored_images[noise_name] = {}
    
    # Mean filter
    start = time.time()
    restored_images[noise_name]["Mean"] = mean_filter(
        noisy_img, 
        kernel_size=3, 
        padding_mode=padding_used
    )
    end = time.time()
    execution_times.append([noise_name, "Mean", end - start])
    
    # Median filter
    start = time.time()
    restored_images[noise_name]["Median"] = median_filter(
        noisy_img, 
        kernel_size=3, 
        padding_mode=padding_used
    )
    end = time.time()
    execution_times.append([noise_name, "Median", end - start])
    
    # Gaussian filter
    start = time.time()
    restored_images[noise_name]["Gaussian"] = gaussian_filter(
        noisy_img, 
        kernel_size=5, 
        sigma=1.0, 
        padding_mode=padding_used
    )
    end = time.time()
    execution_times.append([noise_name, "Gaussian", end - start])

execution_time_df = pd.DataFrame(
    execution_times, 
    columns=["Noise Type", "Filter", "Execution Time"]
)

execution_time_df

### Visualize Restoration Results

In [ ]:
for noise_name, noisy_img in noisy_images.items():
    plt.figure(figsize=(16, 4))
    
    plt.subplot(1, 5, 1)
    plt.imshow(original, cmap="gray", vmin=0, vmax=255)
    plt.title("Original")
    plt.axis("off")
    
    plt.subplot(1, 5, 2)
    plt.imshow(noisy_img, cmap="gray", vmin=0, vmax=255)
    plt.title(noise_name)
    plt.axis("off")
    
    filter_names = ["Mean", "Median", "Gaussian"]
    
    for idx, filter_name in enumerate(filter_names, start=3):
        plt.subplot(1, 5, idx)
        plt.imshow(restored_images[noise_name][filter_name], cmap="gray", vmin=0, vmax=255)
        plt.title(filter_name)
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()


## 2.3 Quantitative Evaluation
### Part A: MSE Function

In [ ]:
def calculate_mse(original, restored):
    """
    Calculates Mean Squared Error manually.
    
    MSE = 1 / MN * ΣΣ [I_original(i,j) - I_restored(i,j)]²
    """
    original = original.astype(np.float64)
    restored = restored.astype(np.float64)
    
    H, W = original.shape
    squared_error_sum = 0.0
    
    for i in range(H):
        for j in range(W):
            diff = original[i, j] - restored[i, j]
            squared_error_sum += diff ** 2
    
    mse = squared_error_sum / (H * W)
    return mse


### Part B: SSIM Function
This implementation uses the manual convolution function and an 11 × 11 uniform window.

In [ ]:
def calculate_ssim(original, restored, window_size=11, padding_mode="mirror"):
    """
    Calculates SSIM manually using local statistics computed by convolution.
    
    SSIM(x,y) = 
    ((2 μx μy + C1)(2 σxy + C2)) /
    ((μx² + μy² + C1)(σx² + σy² + C2))
    """
    original = original.astype(np.float64)
    restored = restored.astype(np.float64)
    
    L = 255
    k1 = 0.01
    k2 = 0.03
    
    C1 = (k1 * L) ** 2
    C2 = (k2 * L) ** 2
    
    # Uniform window
    window = np.ones((window_size, window_size), dtype=np.float64)
    window = window / (window_size * window_size)
    
    # Local means
    mu_x = manual_convolution(original, window, padding_mode=padding_mode)
    mu_y = manual_convolution(restored, window, padding_mode=padding_mode)
    
    # Local squared means
    mu_x_sq = mu_x ** 2
    mu_y_sq = mu_y ** 2
    mu_xy = mu_x * mu_y
    
    # Local second moments
    sigma_x_sq = manual_convolution(original ** 2, window, padding_mode=padding_mode) - mu_x_sq
    sigma_y_sq = manual_convolution(restored ** 2, window, padding_mode=padding_mode) - mu_y_sq
    sigma_xy = manual_convolution(original * restored, window, padding_mode=padding_mode) - mu_xy
    
    numerator = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
    denominator = (mu_x_sq + mu_y_sq + C1) * (sigma_x_sq + sigma_y_sq + C2)
    
    ssim_map = numerator / denominator
    
    return np.mean(ssim_map)


### Generate MSE and SSIM Table

In [ ]:
results = []

for noise_name, noisy_img in noisy_images.items():
    # First evaluate noisy image itself
    noisy_mse = calculate_mse(original, noisy_img)
    noisy_ssim = calculate_ssim(original, noisy_img, window_size=11, padding_mode=padding_used)
    
    results.append([
        noise_name,
        "No Filter",
        noisy_mse,
        noisy_ssim
    ])
    
    # Then evaluate restored images
    for filter_name, restored_img in restored_images[noise_name].items():
        mse = calculate_mse(original, restored_img)
        ssim = calculate_ssim(original, restored_img, window_size=11, padding_mode=padding_used)
        
        results.append([
            noise_name,
            filter_name,
            mse,
            ssim
        ])

results_df = pd.DataFrame(
    results,
    columns=["Noise Type", "Filter", "MSE", "SSIM"]
)

results_df

### Best Filter Per Noise Type

In [ ]:
best_by_mse = results_df.loc[results_df.groupby("Noise Type")["MSE"].idxmin()]
best_by_ssim = results_df.loc[results_df.groupby("Noise Type")["SSIM"].idxmax()]

print("Best results according to lowest MSE:")
display(best_by_mse)

print("Best results according to highest SSIM:")
display(best_by_ssim)


## Execution Time Plots
### Execution Time Table

In [ ]:
execution_time_df

### Execution Time Plot

In [ ]:
plt.figure(figsize=(12, 6))

labels = execution_time_df["Noise Type"] + " - " + execution_time_df["Filter"]
times = execution_time_df["Execution Time"]

plt.bar(range(len(times)), times)
plt.xticks(range(len(times)), labels, rotation=90)
plt.ylabel("Execution Time in Seconds")
plt.title("Execution Time of Restoration Filters")
plt.tight_layout()
plt.show()

## MSE and SSIM Plots
### MSE Plot

In [ ]:
plt.figure(figsize=(14, 6))

labels = results_df["Noise Type"] + " - " + results_df["Filter"]
mse_values = results_df["MSE"]

plt.bar(range(len(mse_values)), mse_values)
plt.xticks(range(len(mse_values)), labels, rotation=90)
plt.ylabel("MSE")
plt.title("MSE Comparison Across Noise Types and Filters")
plt.tight_layout()
plt.show()


### SSIM Plot

In [ ]:
plt.figure(figsize=(14, 6))

labels = results_df["Noise Type"] + " - " + results_df["Filter"]
ssim_values = results_df["SSIM"]

plt.bar(range(len(ssim_values)), ssim_values)
plt.xticks(range(len(ssim_values)), labels, rotation=90)
plt.ylabel("SSIM")
plt.title("SSIM Comparison Across Noise Types and Filters")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Save all images and plot figures for LaTeX report
# ============================================================

import os

OUTPUT_DIR = "latex_report_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_grayscale_image(image, filename):
    """
    Saves a grayscale image as PNG.
    Pixel values are clipped to [0, 255] and converted to uint8.
    """
    image_uint8 = np.clip(image, 0, 255).astype(np.uint8)
    Image.fromarray(image_uint8).save(os.path.join(OUTPUT_DIR, filename))


def safe_filename(name):
    """
    Converts experiment names into clean file names.
    """
    name = name.lower()
    name = name.replace(" ", "_")
    name = name.replace("&", "and")
    name = name.replace(".", "")
    name = name.replace("__", "_")
    return name


# ------------------------------------------------------------
# 1. Save original image
# ------------------------------------------------------------

save_grayscale_image(original, "original_image.png")


# ------------------------------------------------------------
# 2. Save noisy images
# ------------------------------------------------------------

for noise_name, noisy_img in noisy_images.items():
    filename = safe_filename(noise_name) + ".png"
    save_grayscale_image(noisy_img, filename)


# ------------------------------------------------------------
# 3. Save restored images
# ------------------------------------------------------------

for noise_name, filters_dict in restored_images.items():
    for filter_name, restored_img in filters_dict.items():
        filename = safe_filename(noise_name) + "_" + safe_filename(filter_name) + ".png"
        save_grayscale_image(restored_img, filename)


# ------------------------------------------------------------
# 4. Save combined visualization figures
# ------------------------------------------------------------

# Combined noisy image overview
plt.figure(figsize=(15, 8))

plt.subplot(2, 4, 1)
plt.imshow(original, cmap="gray", vmin=0, vmax=255)
plt.title("Original")
plt.axis("off")

for idx, (name, img) in enumerate(noisy_images.items(), start=2):
    plt.subplot(2, 4, idx)
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "all_noisy_images_overview.png"), dpi=300, bbox_inches="tight")
plt.show()


# Save one restoration comparison figure for each noise type
for noise_name, noisy_img in noisy_images.items():
    plt.figure(figsize=(16, 4))
    
    plt.subplot(1, 5, 1)
    plt.imshow(original, cmap="gray", vmin=0, vmax=255)
    plt.title("Original")
    plt.axis("off")
    
    plt.subplot(1, 5, 2)
    plt.imshow(noisy_img, cmap="gray", vmin=0, vmax=255)
    plt.title(noise_name)
    plt.axis("off")
    
    filter_names = ["Mean", "Median", "Gaussian"]
    
    for idx, filter_name in enumerate(filter_names, start=3):
        plt.subplot(1, 5, idx)
        plt.imshow(restored_images[noise_name][filter_name], cmap="gray", vmin=0, vmax=255)
        plt.title(filter_name)
        plt.axis("off")
    
    plt.tight_layout()
    
    filename = safe_filename(noise_name) + "_restoration_comparison.png"
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------
# 5. Save execution time plot
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

labels = execution_time_df["Noise Type"] + " - " + execution_time_df["Filter"]
times = execution_time_df["Execution Time"]

plt.bar(range(len(times)), times)
plt.xticks(range(len(times)), labels, rotation=90)
plt.ylabel("Execution Time in Seconds")
plt.title("Execution Time of Restoration Filters")
plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, "execution_time_plot.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 6. Save MSE plot
# ------------------------------------------------------------

plt.figure(figsize=(14, 6))

labels = results_df["Noise Type"] + " - " + results_df["Filter"]
mse_values = results_df["MSE"]

plt.bar(range(len(mse_values)), mse_values)
plt.xticks(range(len(mse_values)), labels, rotation=90)
plt.ylabel("MSE")
plt.title("MSE Comparison Across Noise Types and Filters")
plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, "mse_plot.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 7. Save SSIM plot
# ------------------------------------------------------------

plt.figure(figsize=(14, 6))

labels = results_df["Noise Type"] + " - " + results_df["Filter"]
ssim_values = results_df["SSIM"]

plt.bar(range(len(ssim_values)), ssim_values)
plt.xticks(range(len(ssim_values)), labels, rotation=90)
plt.ylabel("SSIM")
plt.title("SSIM Comparison Across Noise Types and Filters")
plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, "ssim_plot.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 8. Save CSV tables for report
# ------------------------------------------------------------

results_df.to_csv(os.path.join(OUTPUT_DIR, "mse_ssim_results.csv"), index=False)
execution_time_df.to_csv(os.path.join(OUTPUT_DIR, "execution_time_results.csv"), index=False)

if "edge_results_df" in globals():
    edge_results_df.to_csv(os.path.join(OUTPUT_DIR, "edge_metrics_results.csv"), index=False)


print("All images, plots, and result tables were saved in:")
print(os.path.abspath(OUTPUT_DIR))